### Maussteuerung
Die für die Steuerung zuständige Komponente nennen wir Controller.
Der Controller nimmt Benutzerinput entgegen und ruft dann die entsprechenden Funktionen
der Spielkomponente auf.  

Wir benutzen ein Canvas-Widget, um auf Mausklicks zu hören.

In [1]:
import widget_helpers as W
import grid_helpers as G
import tiktak_L19 as game
from IPython.display import display


grid_spec = (20, 20, 20, 20, 3, 3)
out = W.get_out()
canvas = W.get_canvas()


@out.capture(clear_output=True)
def on_mouse_down(x, y):
    pos = G.xy2cr(x, y, grid_spec, strict=True)
    if pos is None:
        return
    game.play(pos[1]*3 + pos[0])


@out.capture(clear_output=True)
def on_key_down(key, *flags):
    if key == 'n':
        game.new_game()


canvas.on_mouse_down(on_mouse_down)
canvas.on_key_down(on_key_down)
G.draw_grid(canvas, grid_spec, color='blue')
display(canvas, out)
canvas.focus()

Canvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right=…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

***
Wir überschreiben die Funktion `game.update`, so dass zusätzlich eine
textliche Darstellung des Spielfelds ins Output-Widget geschrieben wird.
***

In [2]:
def update(event, **kwargs):
    print(f'{event}, {kwargs}')
    game.show()


game.update = update

***
Wir überschreiben die Funktion `game.update`, so dass
das Spielfeld graphisch auf der Leinwand dargestellt wird.  

Gewinnt ein Spieler, markieren wir die Felder mit dem TikTakToe.
Damit der ursprüngliche Inhalt des Feldes sichtbar bleibt, 
setzten wir  
`canvas.global_alpha = 0.3`, was die Feldmarkierung etwas transparent macht.

Die Funktion modifiziert Canvas-Attribute wie `fill_style`,  `global_alpha`, `text_align`, ...
Deshalb sichern wir zu Beginn den Zustand der Canvas mit
`canvas.save()` und stellen diesen ZUstand am Schluss mit
`canvas.restore()` wieder her.  
Siehe [https://ipycanvas.readthedocs.io/en/latest/canvas_state.html](https://ipycanvas.readthedocs.io/en/latest/canvas_state.html)
***

In [10]:
def update(event, **kwargs):
    canvas.save()
    if event == 'new_game':
        canvas.clear()
        G.draw_grid(canvas, grid_spec, color='blue')
    if event == 'play':
        G.fill_text(canvas, kwargs['player'], kwargs['pos'], grid_spec)
    if event == 'winner':
        canvas.global_alpha = 0.3
        for pos in kwargs['win_line']:
            G.fill_rect(canvas, pos, grid_spec, color='red')
        canvas.global_alpha = 1
    else:
        print(event, kwargs)
    canvas.restore()


game.update = update